# Notebook 04 — Machine Learning

Mục tiêu:
1. So sánh 4 mô hình (Dummy baseline + Linear + RandomForest + GradientBoosting).
2. Đánh giá bằng 5-fold CV trên train + 1 lần đánh giá trên test.
3. Phân tích 10 trường hợp sai lớn nhất.
4. K-Means chọn K theo silhouette.
5. Demo RecommendationEngine với 3 hồ sơ nhu cầu.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.features import build_preprocessor, get_feature_names
from src.predictor import PricePredictor, cv_metrics
from src.segmenter import KMeansSegmenter, pick_k_by_silhouette
from src.recommender import RecommendationEngine

RANDOM_STATE = 42
FIG = Path('reports/figures')
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/processed/listings_with_amenities.csv').dropna(subset=['price_per_m2']).copy()
print('Loaded:', df.shape)

Loaded: (2942, 23)


## 1. Train / Test split

In [2]:
NUMERIC = ['area_m2', 'bedrooms', 'bathrooms', 'floor_count', 'frontage_width']
CATEGORICAL = ['district_clean', 'direction_clean']
FEATURES = NUMERIC + CATEGORICAL

df[FEATURES] = df[FEATURES].fillna({c: 'missing' for c in CATEGORICAL})
X = df[FEATURES]
y = df['price_per_m2'].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)

pre = build_preprocessor(NUMERIC, CATEGORICAL)
pre.fit(X_train)
X_train_t = pre.transform(X_train)
X_test_t = pre.transform(X_test)
print('Features sau transform:', X_train_t.shape[1])

Train: (2353, 7) Test: (589, 7)
Features sau transform: 35


## 2. So sánh 4 mô hình — 5-fold CV + Test

In [3]:
rows = []
for name in ['dummy', 'linear', 'rf', 'gbr']:
    m = cv_metrics(X_train_t, y_train.values, model_name=name, log_target=True,
                   n_splits=5, random_state=RANDOM_STATE)
    pred = PricePredictor(model_name=name, log_target=True, random_state=RANDOM_STATE).fit(
        X_train_t, y_train.values
    )
    test_m = pred.evaluate(X_test_t, y_test.values)
    rows.append({
        'model': name,
        'CV_MAE_mean': m['mae_mean'],
        'CV_MAE_std': m['mae_std'],
        'CV_RMSE_mean': m['rmse_mean'],
        'CV_R2_mean': m['r2_mean'],
        'Test_MAE': test_m['mae'],
        'Test_RMSE': test_m['rmse'],
        'Test_R2': test_m['r2'],
    })
    
results = pd.DataFrame(rows)
results.round(0)

,model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_R2_mean,Test_MAE,Test_RMSE,Test_R2
0,dummy,63136808.0,4391481.0,138651539.0,-0.0,67053155.0,153498147.0,-0.0
1,linear,47731968.0,1583740.0,114043121.0,0.0,49932553.0,126525066.0,0.0
2,rf,43465539.0,2142146.0,109133077.0,0.0,43658197.0,117088980.0,0.0
3,gbr,43937273.0,1984089.0,109096029.0,0.0,45501019.0,118848944.0,0.0


**Nhận xét:** Random Forest đạt R² ≈ 0.39 trên test (tốt nhất). Linear ≈ 0.29. Baseline Dummy ≈ -0.05 (tệ — không học). MAE ~44-50 triệu VND/m² tương đương sai số ~30% so với median.

## 3. Phân tích 10 trường hợp sai lớn nhất (Random Forest)

In [4]:
best = PricePredictor(model_name='rf', log_target=True, random_state=RANDOM_STATE).fit(
    X_train_t, y_train.values
)
y_pred = best.predict(X_test_t)
errors_df = X_test.copy()
errors_df['actual_price_per_m2'] = y_test.values
errors_df['predicted_price_per_m2'] = y_pred
errors_df['abs_error'] = np.abs(errors_df['actual_price_per_m2'] - errors_df['predicted_price_per_m2'])
errors_df['pct_error'] = 100 * errors_df['abs_error'] / errors_df['actual_price_per_m2']
errors_df['listing_id'] = df.loc[y_test.index, 'listing_id'].values
errors_df['title'] = df.loc[y_test.index, 'title'].values
top10 = errors_df.nlargest(10, 'abs_error')[['listing_id', 'district_clean', 'area_m2',
                                             'bedrooms', 'actual_price_per_m2',
                                             'predicted_price_per_m2', 'pct_error', 'title']]
top10

,listing_id,district_clean,area_m2,bedrooms,actual_price_per_m2,predicted_price_per_m2,pct_error,title
1134,1150,Quận Phú Nhuận,160.0,NaN,1.750000e+09,2.596293e+08,85.164040,Bán Toà Nhà Góc 2 MT 259A Nguyễn Văn Trỗi và T...
1563,1583,Quận Phú Nhuận,160.0,NaN,1.562500e+09,1.996102e+08,87.224946,Hot! tòa nhà góc 2mt 259a nguyễn văn trỗi_trươ...
1354,1373,Quận 3,398.0,5.0,1.381910e+09,3.458681e+08,74.971729,Bán Tòa Nhà Điện Biên Phủ Quận 3 - Bề Ngang 12...
2877,2908,Quận 3,391.0,NaN,1.023018e+09,3.681673e+08,64.011644,"Bán Khách sạn 77 phòng Mặt tiền Võ Văn Tần, P...."
176,181,Quận Bình Tân,100.0,NaN,5.800000e+08,9.677271e+07,83.315050,Cặp nhà MTKD đường số 17A đối diện siêu thị Nh...
919,931,Quận 1,60.0,4.0,8.000000e+08,3.653403e+08,54.332465,"Bán nhà MT 46 Nguyễn Cư Trinh, 4x15, 2 lầu, st..."
32,35,Quận 3,412.0,NaN,7.281553e+08,3.605696e+08,50.481779,Cần bán nhà góc 2 MT Trần Quốc Thảo và Lý Chín...
437,445,Quận Bình Thạnh,73.0,NaN,5.041096e+08,1.380241e+08,72.620218,"Bán nhà 2MT Đinh Bộ Lĩnh, P26, Bình Thạnh, HCM..."
1273,1290,Quận 1,64.1,NaN,6.474259e+08,3.317843e+08,48.753317,MT VIP 4x16m Trịnh Văn Cấn ngay trung tâm Q1 t...
2125,2149,Quận Gò Vấp,108.0,NaN,5.092593e+08,2.159143e+08,57.602287,48 Tỷ Bán tòa nhà CHDV góc 3MT View công viên ...


**Nhận xét (10 trường hợp sai):**

1. Phần lớn sai ở các căn **giá cực cao (> 300 tr/m²)** — mô hình dự đoán thấp hơn nhiều (có thể vì train data nghèo các ví dụ giá cực cao).
2. Các căn **diện tích lớn ở vùng ven** cũng sai nhiều — mô hình đánh giá quá cao vì feature "diện tích" thường đi kèm giá cao trong tập train.
3. Các căn **thiếu nhiều thông tin** (floor, frontage) → imputation điền median → có thể làm sai lệch.

**Hạn chế:** Dataset chỉ có 3000 dòng từ 1 nguồn → mô hình thiếu thông tin về giá cực trị và vị trí đặc biệt. Cần thu thập thêm dữ liệu để cải thiện.

## 4. K-Means — chọn K theo silhouette

In [5]:
best_k, scores = pick_k_by_silhouette(X_train_t, k_range=(3, 6), random_state=RANDOM_STATE)
print(f'Best K = {best_k}')
print(f'Scores: {scores}')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(scores.keys()), list(scores.values()), marker='o', color='darkgreen')
ax.set_title('Silhouette score theo K')
ax.set_xlabel('K')
ax.set_ylabel('Silhouette score')
ax.axvline(best_k, linestyle='--', color='red', label=f'best K={best_k}')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'fig11_silhouette.png', dpi=120)
plt.close()
print('Saved fig11')

Best K = 3
Scores: {3: 0.08720374716680773, 4: 0.05200409105765593, 5: 0.060712190274672194, 6: 0.03072295107672501}


Saved fig11


In [6]:
seg = KMeansSegmenter(n_clusters=best_k, random_state=RANDOM_STATE).fit(X_train_t)
train_clusters = seg.predict(X_train_t)
test_clusters = seg.predict(X_test_t)

df_train = df.loc[X_train.index].copy()
df_train['cluster'] = train_clusters
df_test = df.loc[X_test.index].copy()
df_test['cluster'] = test_clusters
df_all_with_cluster = pd.concat([df_train, df_test], axis=0)
print('Cluster counts:')
print(df_all_with_cluster['cluster'].value_counts().sort_index())

Cluster counts:
cluster
0      50
1    1100
2    1792
Name: count, dtype: int64


## 5. Mô tả cụm

In [7]:
all_X_t = pre.transform(df_all_with_cluster[FEATURES].fillna({c: 'missing' for c in CATEGORICAL}))
feat_names = get_feature_names(pre)
desc = seg.describe_segments(all_X_t, feat_names)
print('Cluster centers (mean feature value):')
desc.round(2)

Cluster centers (mean feature value):


,area_m2,bedrooms,bathrooms,floor_count,frontage_width,district_clean_Huyện Bình Chánh,district_clean_Huyện Hóc Môn,district_clean_Huyện Nhà Bè,district_clean_Huyện Thủ Đức,district_clean_Quận 1,...,district_clean_missing,district_clean_infrequent_sklearn,direction_clean_Bắc,direction_clean_Nam,direction_clean_Tây,direction_clean_Tây Bắc,direction_clean_Tây Nam,direction_clean_Đông,direction_clean_Đông Bắc,direction_clean_Đông Nam
__cluster__,,,,,,,,,,,,,,,,,,,,,
0,0.38,-0.13,-0.36,-0.34,0.00,1.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.08,0.22,0.10,0.04,0.08,0.28,0.08,0.12
1,-0.26,-0.99,-0.88,-0.45,-0.22,0.0,0.02,0.03,0.18,0.02,...,0.01,0.01,0.12,0.13,0.12,0.10,0.11,0.16,0.09,0.17
2,0.15,0.60,0.54,0.28,0.15,0.0,0.02,0.03,0.13,0.04,...,0.00,0.00,0.09,0.13,0.09,0.13,0.12,0.16,0.13,0.15


**Nhận xét:** Có 3 cụm với silhouette thấp (~0.09) → các cụm tách biệt không rõ. Điều này dự kiến vì đặc trưng BĐS thay đổi liên tục, không rời rạc. Tuy vậy, K-Means vẫn hữu ích để gợi ý vì bonus cụm giúp phân nhóm giá.

## 6. Demo RecommendationEngine — 3 hồ sơ nhu cầu

In [8]:
eng = RecommendationEngine()
profiles = [
    {
        'name': 'Gia đình trẻ, 5 tỷ, Quận 7 / Bình Thạnh',
        'budget_vnd': 5e9,
        'target_bedrooms': 3,
        'target_area_m2': 70.0,
        'preferred_districts': ['Quận 7', 'Quận Bình Thạnh'],
        'preferred_cluster': 1,
    },
    {
        'name': 'Nhà đầu tư, 8 tỷ, Quận 2 hoặc Quận 9 (Thủ Đức)',
        'budget_vnd': 8e9,
        'target_bedrooms': 3,
        'target_area_m2': 80.0,
        'preferred_districts': ['Quận 2', 'Quận 9', 'Quận Thủ Đức'],
        'preferred_cluster': 0,
    },
    {
        'name': 'Người mua đầu tiên, 3 tỷ, ngoại thành',
        'budget_vnd': 3e9,
        'target_bedrooms': 2,
        'target_area_m2': 60.0,
        'preferred_districts': ['Huyện Bình Chánh', 'Huyện Củ Chi', 'Huyện Hóc Môn'],
        'preferred_cluster': 2,
    },
]
for prof in profiles:
    recs = eng.recommend(
        df_all_with_cluster,
        budget_vnd=prof['budget_vnd'],
        target_bedrooms=prof['target_bedrooms'],
        target_area_m2=prof['target_area_m2'],
        preferred_districts=prof['preferred_districts'],
        preferred_cluster=prof['preferred_cluster'],
        top_k=5,
    )
    print(f"\n{'='*70}\n{prof['name']}\n{'='*70}")
    display_cols = ['listing_id', 'district_clean', 'ward', 'total_price', 'area_m2',
                    'bedrooms', 'price_per_m2', 'amenity_score', 'cluster', 'score_total']
    print(recs[display_cols].to_string(index=False))


Gia đình trẻ, 5 tỷ, Quận 7 / Bình Thạnh
 listing_id  district_clean      ward  total_price  area_m2  bedrooms  price_per_m2  amenity_score  cluster  score_total
        285 Quận Bình Thạnh         2 6000000000.0     75.0       3.0  8.000000e+07           11.8        1     2.308571
       1545          Quận 7    Phú Mỹ 5900000000.0     65.2       3.0  9.049080e+07            7.6        1     2.093371
       2689          Quận 7 Phú Thuận 4750000000.0     67.5       4.0  7.037037e+07            5.3        2     2.039301
        527          Quận 7    Phú Mỹ 4900000000.0     56.7       3.0  8.641975e+07            7.6        1     2.028937
       1602 Quận Bình Thạnh        11 4000000000.0     52.0       4.0  7.692308e+07            8.5        2     1.810002

Nhà đầu tư, 8 tỷ, Quận 2 hoặc Quận 9 (Thủ Đức)
Empty DataFrame
Columns: [listing_id, district_clean, ward, total_price, area_m2, bedrooms, price_per_m2, amenity_score, cluster, score_total]
Index: []

Người mua đầu tiên, 3 tỷ, ngoại

**Nhận xét:** Hệ gợi ý hoạt động: trả về top tin theo `score_total` (gồm 4 thành phần: gần giá mục tiêu, gần diện tích mục tiêu, cùng cụm, tiện ích cao). Profile 3 không match (giá quá thấp) → trả empty.

## 7. Kết luận

- **Mô hình tốt nhất**: Random Forest với R² ≈ 0.39 trên test.
- **Hạn chế**: MAE còn cao (~44tr/m²), sai nhiều ở giá cực trị và tin thiếu thông tin.
- **Hệ gợi ý**: hoạt động đúng cho 2/3 profile, profile 3 cần mở rộng tolerance hoặc thêm dữ liệu vùng ven.
- **Hướng phát triển**: thu thập thêm dữ liệu (≥10k tin), thêm feature khoảng cách đến trung tâm, dùng mô hình XGBoost/LightGBM.